# Qualidade de `eventos_assinatura.csv`

Diagnóstico do log de eventos de assinatura por cinco critérios:

| Critério | Pergunta |
|---|---|
| **Completude** | Está tudo lá? Campos preenchidos e eventos que deveriam existir |
| **Validade** | Os valores estão no formato e no domínio esperados? |
| **Consistência** | Os dados concordam entre si, dentro do arquivo e com os outros arquivos? |
| **Unicidade** | Cada acontecimento aparece uma vez só? |
| **Temporalidade** | As datas estão no período certo, sem buracos, no ritmo esperado? |

**Só diagnóstico:** nada é corrigido aqui. Cada achado vai para uma tabela-resumo em `saidas/`, com o tratamento sugerido.
Nada em `dados/brutos/` é alterado: o hash dos arquivos é conferido no início e no fim.

**Fuso:** pelo dicionário, as lojas gravam em UTC (terminado em `Z`) e o site em horário de Brasília.
Para decidir em que dia cai cada evento, tudo é convertido para o horário de Brasília (UTC−3, sem horário de verão desde 2019),
que é o fuso do corte.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
SAIDAS = ROOT / "saidas"
INICIO_LOG = pd.Timestamp("2024-07-01")
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)  # 01/10/2026 00:00, horário de Brasília
TZ_BR = "America/Sao_Paulo"

ARQUIVOS = ["eventos_assinatura.csv", "assinaturas.csv", "usuarios.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

achados = []

def achado(criterio, checagem, resultado, linhas, tratamento, status="atenção"):
    """status: ok (nada encontrado), esperado (encontrado, mas é comportamento do negócio) ou atenção."""
    achados.append({"criterio": criterio, "checagem": checagem, "status": "ok" if linhas == 0 else status,
                    "resultado": resultado, "linhas_afetadas": int(linhas), "tratamento_sugerido": tratamento})

## 0. Leitura

Tudo como texto e sem converter vazio em nulo, para ver o arquivo exatamente como chegou.

In [2]:
eventos = pd.read_csv(BRUTOS / "eventos_assinatura.csv", dtype=str, keep_default_na=False)
assinaturas = pd.read_csv(BRUTOS / "assinaturas.csv", dtype=str)
usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, usecols=["user_id", "email"])

assert len(eventos) == 275_565  # linhas segundo o dicionário
N = len(eventos)
print(f"eventos_assinatura: {N:,} linhas x {eventos.shape[1]} colunas -> {list(eventos.columns)}")
eventos.sample(5, random_state=0)

eventos_assinatura: 275,565 linhas x 6 colunas -> ['user_id', 'data_hora', 'evento', 'plano', 'canal_cobranca', 'detalhe']


,user_id,data_hora,evento,plano,canal_cobranca,detalhe
11777,u_01b39575,2024-08-27 22:03:03,renovacao,mensal,web_pix,
78453,u_3d07651d,2025-04-18T01:54:27Z,renovacao,mensal,google_play,
102322,u_bee2f014,2025-07-21 08:39:54,chargeback,anual,web_cartao,
194874,u_2a39e9ff,2026-03-08T14:38:51Z,assinatura_expirada,anual,app_store,
62018,u_bd664ac9,2025-02-25T00:11:33Z,renovacao,mensal,google_play,


### Datas em UTC e em horário de Brasília

As duas colunas abaixo são **derivadas** para a análise; o arquivo original não muda.

In [3]:
eh_utc = eventos["data_hora"].str.endswith("Z")
dh_utc = pd.Series(pd.NaT, index=eventos.index, dtype="datetime64[ns, UTC]")
dh_utc[eh_utc] = pd.to_datetime(eventos.loc[eh_utc, "data_hora"], format="%Y-%m-%dT%H:%M:%SZ", errors="coerce", utc=True)
dh_utc[~eh_utc] = (pd.to_datetime(eventos.loc[~eh_utc, "data_hora"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
                   .dt.tz_localize(TZ_BR).dt.tz_convert("UTC"))

eventos["formato"] = eh_utc.map({True: "UTC (Z)", False: "Brasília"})
eventos["dh_utc"] = dh_utc
eventos["dh_br"] = dh_utc.dt.tz_convert(TZ_BR).dt.tz_localize(None)
eventos["data_br"] = eventos["dh_br"].dt.normalize()
eventos["mes_br"] = eventos["dh_br"].dt.to_period("M")
eventos[["data_hora", "formato", "dh_br"]].drop_duplicates("formato")

,data_hora,formato,dh_br
0,2024-07-01T22:22:27Z,UTC (Z),2024-07-01 19:22:27
2,2024-07-01 08:42:40,Brasília,2024-07-01 08:42:40


## 1. Completude

Duas perguntas: os campos estão preenchidos? E os eventos que deveriam existir existem?

In [4]:
VAZIOS_DISFARCADOS = {"nan", "null", "none", "na", "n/a", "-"}
colunas = ["user_id", "data_hora", "evento", "plano", "canal_cobranca", "detalhe"]
vazios = pd.DataFrame({
    "vazio": [(eventos[c].str.strip() == "").sum() for c in colunas],
    "vazio_disfarcado": [eventos[c].str.strip().str.lower().isin(VAZIOS_DISFARCADOS).sum() for c in colunas],
}, index=colunas)
vazios["% vazio"] = (100 * vazios["vazio"] / N).round(1)
vazios

,vazio,vazio_disfarcado,% vazio
user_id,0,0,0.0
data_hora,0,0,0.0
evento,0,0,0.0
plano,72873,0,26.4
canal_cobranca,0,0,0.0
detalhe,269494,0,97.8


In [5]:
eventos["plano_vazio"] = eventos["plano"] == ""
eventos["detalhe_preenchido"] = eventos["detalhe"] != ""
pd.crosstab(eventos["evento"], [eventos["plano_vazio"].rename("plano vazio")], margins=True)

plano vazio,False,True,All
evento,,,
assinatura_expirada,17830,0,17830
assinatura_iniciada,27383,0,27383
cancelamento_web,1172,0,1172
chargeback,296,0,296
periodo_carencia,1346,0,1346
reembolso,72,0,72
renovacao,132606,0,132606
renovacao_automatica_desligada,20189,6734,26923
renovacao_automatica_religada,1798,0,1798


In [6]:
eh_trial_nome = eventos["evento"].str.startswith("trial_")
plano_vazio_fora_trial = eventos["plano_vazio"] & ~eh_trial_nome
plano_cheio_em_trial = (~eventos["plano_vazio"] & eh_trial_nome).sum()
print(f"plano vazio em evento que não começa com trial_: {plano_vazio_fora_trial.sum():,}")
print("  quais eventos:", eventos.loc[plano_vazio_fora_trial, "evento"].value_counts().to_dict())
print(f"plano preenchido em evento de trial: {plano_cheio_em_trial:,}")

plano vazio em evento que não começa com trial_: 6,734
  quais eventos: {'renovacao_automatica_desligada': 6734}
plano preenchido em evento de trial: 0


Todo `plano` vazio fora dos eventos `trial_*` é `renovacao_automatica_desligada`. A hipótese é que seja gente desligando
a renovação **durante o teste grátis**, antes de existir plano. A célula abaixo confere onde esses eventos caem.

In [7]:
ordem_ev = eventos.sort_values(["user_id", "dh_utc"], kind="stable")
ordem_ev["anterior"] = ordem_ev.groupby("user_id")["evento"].shift()
ordem_ev["seguinte"] = ordem_ev.groupby("user_id")["evento"].shift(-1)
ultimo_trial = ordem_ev["dh_utc"].where(ordem_ev["evento"] == "trial_iniciado").groupby(ordem_ev["user_id"]).ffill()
ordem_ev["dias_desde_trial"] = (ordem_ev["dh_utc"] - ultimo_trial).dt.total_seconds() / 86400
dt = ordem_ev.loc[plano_vazio_fora_trial[ordem_ev.index]]
print("evento anterior:", dt["anterior"].value_counts(dropna=False).to_dict())
print("evento seguinte:", dt["seguinte"].fillna("(nenhum: trial em andamento)").value_counts().to_dict())
print("dias desde o início do trial:", dt["dias_desde_trial"].describe().round(2).to_dict())

desligada_no_trial = plano_vazio_fora_trial & (ordem_ev["dias_desde_trial"].reindex(eventos.index) <= 8)
achado("Completude", "plano vazio em evento que não é trial_*",
       f"{plano_vazio_fora_trial.sum():,} linhas, todas renovacao_automatica_desligada até 8 dias após o início do trial "
       f"({desligada_no_trial.sum():,}): a pessoa desliga a renovação durante o teste grátis",
       plano_vazio_fora_trial.sum(),
       "Não é erro. Tratar como evento do trial (cancelamento no teste), não da assinatura", status="esperado")

# Daqui em diante, 'evento de trial' inclui o desligamento durante o teste
eh_trial = eh_trial_nome | desligada_no_trial

evento anterior: {'trial_iniciado': 6200, 'trial_expirado': 534}
evento seguinte: {'trial_expirado': 6178, '(nenhum: trial em andamento)': 556}
dias desde o início do trial: {'count': 6734.0, 'mean': 4.5, 'std': 1.72, 'min': 1.3, '25%': 3.0, '50%': 4.49, '75%': 5.98, 'max': 7.67}


In [8]:
# detalhe: em que eventos aparece?
pd.crosstab(eventos["evento"], eventos["detalhe"].replace("", "(vazio)"))

detalhe,(vazio),chargeback,falha_pagamento,reembolso,renovacao_desligada
evento,,,,,
assinatura_expirada,11759,109,800,23,5139
assinatura_iniciada,27383,0,0,0,0
cancelamento_web,1172,0,0,0,0
chargeback,296,0,0,0,0
periodo_carencia,1346,0,0,0,0
reembolso,72,0,0,0,0
renovacao,132606,0,0,0,0
renovacao_automatica_desligada,26923,0,0,0,0
renovacao_automatica_religada,1798,0,0,0,0


In [9]:
# Na expiração, o detalhe deveria trazer o motivo. Quanto falta, comparado ao motivo_fim de assinaturas.csv?
exp = eventos[eventos["evento"] == "assinatura_expirada"]
exp_sem_detalhe = (exp["detalhe"] == "").sum()
print(f"assinatura_expirada: {len(exp):,} | sem detalhe: {exp_sem_detalhe:,} ({exp_sem_detalhe / len(exp):.1%})")
print("motivo_fim em assinaturas.csv:", assinaturas["motivo_fim"].value_counts().to_dict())
print("detalhe em assinatura_expirada:", exp["detalhe"].replace("", "(vazio)").value_counts().to_dict())
achado("Completude", "detalhe (motivo) ausente em assinatura_expirada",
       f"{exp_sem_detalhe:,} de {len(exp):,} expirações sem motivo ({exp_sem_detalhe / len(exp):.0%}); "
       "assinaturas.csv tem motivo_fim em todas",
       exp_sem_detalhe,
       "Não usar detalhe como fonte de motivo; usar motivo_fim de assinaturas.csv (casando por user_id e data)")

assinatura_expirada: 17,830 | sem detalhe: 11,759 (66.0%)
motivo_fim em assinaturas.csv: {'renovacao_desligada': 16145, 'falha_pagamento': 1321, 'chargeback': 292, 'reembolso': 72}
detalhe em assinatura_expirada: {'(vazio)': 11759, 'renovacao_desligada': 5139, 'falha_pagamento': 800, 'chargeback': 109, 'reembolso': 23}


### Cobertura: os eventos que deveriam existir existem?

Não há `assinatura_id` nos eventos, então o casamento com `assinaturas.csv` é por `user_id` + data.
O histórico começa em 01/07/2024: só assinaturas iniciadas a partir daí devem ter `assinatura_iniciada`.
Como o horário pode cair na virada do dia, aceita-se diferença de até 1 dia.

In [10]:
eh_teste = usuarios.set_index("user_id")["email"].str.strip().str.lower().str.endswith("@vitaliza.com.br")
assinaturas["teste"] = assinaturas["user_id"].map(eh_teste)
assinaturas["inicio"] = pd.to_datetime(assinaturas["inicio"], format="%Y-%m-%d")
assinaturas["data_fim"] = pd.to_datetime(assinaturas["data_fim"], format="%Y-%m-%d")
ass = assinaturas[~assinaturas["teste"]].copy()

def casa_por_data(base, col_data, evento, tolerancia_dias=1):
    """Para cada linha de base, diz se o user_id tem o evento a até N dias da data."""
    ev = eventos.loc[eventos["evento"] == evento, ["user_id", "data_br"]]
    m = base[["assinatura_id", "user_id", col_data]].merge(ev, on="user_id", how="left")
    m["ok"] = (m["data_br"] - m[col_data]).dt.days.abs() <= tolerancia_dias
    return m.groupby("assinatura_id")["ok"].any()

no_periodo = ass[ass["inicio"] >= INICIO_LOG]
tem_inicio = casa_por_data(no_periodo, "inicio", "assinatura_iniciada")
encerradas = ass[ass["data_fim"].notna()]
tem_fim = casa_por_data(encerradas, "data_fim", "assinatura_expirada")

cobertura = pd.DataFrame({
    "assinaturas esperadas": [len(no_periodo), len(encerradas)],
    "com o evento": [tem_inicio.sum(), tem_fim.sum()],
    "sem o evento": [(~tem_inicio).sum(), (~tem_fim).sum()],
}, index=["assinatura_iniciada (início a partir de 01/07/2024)", "assinatura_expirada (com data_fim)"])
cobertura

,assinaturas esperadas,com o evento,sem o evento
assinatura_iniciada (início a partir de 01/07/2024),27383,27383,0
assinatura_expirada (com data_fim),17830,17830,0


In [11]:
# Sentido inverso: evento de início que não corresponde a nenhuma assinatura de assinaturas.csv
ini_ev = eventos.loc[eventos["evento"] == "assinatura_iniciada", ["user_id", "data_br"]].reset_index()
m = ini_ev.merge(assinaturas[["user_id", "inicio"]], on="user_id", how="left")
m["ok"] = (m["data_br"] - m["inicio"]).dt.days.abs() <= 1
ini_sem_assinatura = (~m.groupby("index")["ok"].any()).sum()
print(f"assinatura_iniciada sem assinatura correspondente em assinaturas.csv: {ini_sem_assinatura:,}")

sem_ini, sem_fim = int((~tem_inicio).sum()), int((~tem_fim).sum())
achado("Completude", "assinaturas sem evento de início (iniciadas a partir de 01/07/2024)",
       f"{sem_ini:,} de {len(no_periodo):,}", sem_ini,
       "Se houver: reconstruir o início pela data de inicio de assinaturas.csv e marcar a origem")
achado("Completude", "assinaturas encerradas sem evento de expiração",
       f"{sem_fim:,} de {len(encerradas):,}", sem_fim,
       "Se houver: usar data_fim de assinaturas.csv como fim")
achado("Completude", "evento de início sem assinatura em assinaturas.csv",
       f"{ini_sem_assinatura:,} eventos", ini_sem_assinatura,
       "Se houver: investigar antes de usar; podem ser assinaturas apagadas do retrato")

assinatura_iniciada sem assinatura correspondente em assinaturas.csv: 0


In [12]:
# Contas de teste: aparecem no log?
eventos["teste"] = eventos["user_id"].map(eh_teste)
ev_teste = int(eventos["teste"].fillna(False).sum())
print(f"eventos de contas de teste (@vitaliza.com.br): {ev_teste:,}")
achado("Completude", "eventos de contas de teste", f"{ev_teste:,} eventos", ev_teste,
       "Se houver: excluir, como em assinaturas.csv")

eventos de contas de teste (@vitaliza.com.br): 0


## 2. Validade

Cada valor está no formato e na lista de valores que o dicionário prevê?

In [13]:
DOMINIO = {
    "evento": {"trial_iniciado", "trial_expirado", "assinatura_iniciada", "renovacao",
               "renovacao_automatica_desligada", "renovacao_automatica_religada", "periodo_carencia",
               "assinatura_expirada", "cancelamento_web", "reembolso", "chargeback"},
    "plano": {"mensal", "anual", ""},
    "canal_cobranca": {"app_store", "google_play", "web_cartao", "web_pix"},
}
for col, dominio in DOMINIO.items():
    fora = ~eventos[col].isin(dominio)
    print(f"{col}: {fora.sum():,} fora do domínio {sorted(eventos.loc[fora, col].unique())[:10]}")
    achado("Validade", f"{col} fora da lista do dicionário", f"{fora.sum():,} linhas", fora.sum(),
           "Nenhum se for zero; se não, mapear para o valor do dicionário")

# Maiúsculas e espaços que passariam despercebidos
for col in ["evento", "plano", "canal_cobranca", "detalhe", "user_id"]:
    sujo = (eventos[col] != eventos[col].str.strip().str.lower()).sum()
    print(f"{col}: {sujo:,} valores com maiúscula ou espaço nas pontas")

evento: 0 fora do domínio []
plano: 0 fora do domínio []
canal_cobranca: 0 fora do domínio []
evento: 0 valores com maiúscula ou espaço nas pontas
plano: 0 valores com maiúscula ou espaço nas pontas


canal_cobranca: 0 valores com maiúscula ou espaço nas pontas
detalhe: 0 valores com maiúscula ou espaço nas pontas
user_id: 0 valores com maiúscula ou espaço nas pontas


In [14]:
# detalhe: o dicionário não lista os valores; comparamos com o domínio de motivo_fim
DOMINIO_DETALHE = set(assinaturas["motivo_fim"].dropna()) | {""}
fora = ~eventos["detalhe"].isin(DOMINIO_DETALHE)
print("valores de detalhe:", eventos["detalhe"].replace("", "(vazio)").value_counts().to_dict())
achado("Validade", "detalhe fora dos motivos de assinaturas.csv", f"{fora.sum():,} linhas", fora.sum(),
       "O dicionário não lista os valores de detalhe; confirmar com o dono do dado se aparecer valor novo")

valores de detalhe: {'(vazio)': 269494, 'renovacao_desligada': 5139, 'falha_pagamento': 800, 'chargeback': 109, 'reembolso': 23}


In [15]:
# data_hora: os dois formatos convertem? Há data impossível?
padrao = eventos["data_hora"].str.replace(r"\d", "9", regex=True)
print("padrões de data_hora:", padrao.value_counts().to_dict())
nao_converte = eventos["dh_utc"].isna().sum()
print(f"datas que não convertem: {nao_converte:,}")
achado("Validade", "data_hora que não converte em data", f"{nao_converte:,} linhas; dois formatos: "
       f"{(eventos['formato'] == 'UTC (Z)').sum():,} em UTC e {(eventos['formato'] == 'Brasília').sum():,} em Brasília",
       nao_converte, "Converter cada formato com seu fuso para uma coluna única em horário de Brasília")

padrões de data_hora: {'9999-99-99T99:99:99Z': 186323, '9999-99-99 99:99:99': 89242}
datas que não convertem: 0


In [16]:
# user_id: formato e existência em usuarios.csv
fora_padrao = ~eventos["user_id"].str.fullmatch(r"u_[0-9a-f]{8}")
sem_cadastro = ~eventos["user_id"].isin(usuarios["user_id"])
print(f"user_id fora do padrão u_xxxxxxxx: {fora_padrao.sum():,} | sem cadastro em usuarios.csv: {sem_cadastro.sum():,}")
achado("Validade", "user_id fora do padrão ou sem cadastro", f"{fora_padrao.sum():,} fora do padrão; {sem_cadastro.sum():,} sem cadastro",
       (fora_padrao | sem_cadastro).sum(), "Se houver: excluir e registrar")

user_id fora do padrão u_xxxxxxxx: 0 | sem cadastro em usuarios.csv: 0


## 3. Consistência

### 3.1 Formato da data × canal

O dicionário diz que as lojas mandam em UTC e o site grava em horário de Brasília.

In [17]:
formato_canal = pd.crosstab(eventos["canal_cobranca"], eventos["formato"])
display(formato_canal)
LOJAS = {"app_store", "google_play"}
eh_loja = eventos["canal_cobranca"].isin(LOJAS)
formato_trocado = ((eh_loja & (eventos["formato"] != "UTC (Z)")) | (~eh_loja & (eventos["formato"] != "Brasília"))).sum()
achado("Consistência", "formato de data diferente do esperado para o canal",
       f"{formato_trocado:,} linhas (loja sem Z ou site com Z)", formato_trocado,
       "Nenhum se for zero; se não, decidir o fuso pelo formato da linha, não pelo canal")

formato,Brasília,UTC (Z)
canal_cobranca,,
app_store,0,92741
google_play,0,93582
web_cartao,58775,0
web_pix,30467,0


### 3.2 Evento × canal

`cancelamento_web` só deveria existir no site. Na loja, o cancelamento chega como `renovacao_automatica_desligada`.

In [18]:
evento_canal = pd.crosstab(eventos["evento"], eventos["canal_cobranca"])
display(evento_canal)
cancel_web_na_loja = ((eventos["evento"] == "cancelamento_web") & eh_loja).sum()
achado("Consistência", "cancelamento_web em canal de loja", f"{cancel_web_na_loja:,} linhas", cancel_web_na_loja,
       "Nenhum se for zero; se não, tratar como renovacao_automatica_desligada")

canal_cobranca,app_store,google_play,web_cartao,web_pix
evento,,,,
assinatura_expirada,5775,5984,4093,1978
assinatura_iniciada,9458,9140,5939,2846
cancelamento_web,0,0,774,398
chargeback,96,89,70,41
periodo_carencia,257,277,683,129
reembolso,21,28,16,7
renovacao,46208,44535,26522,15341
renovacao_automatica_desligada,8813,9546,5737,2827
renovacao_automatica_religada,610,640,394,154


### 3.3 Ordem dos eventos por usuário

Para cada usuário, os eventos de assinatura (sem os de trial, incluindo o desligamento durante o teste) são ordenados no tempo e acompanha-se se ele tem assinatura ativa:
`assinatura_iniciada` abre, `assinatura_expirada` fecha. Quem tem renovação antes de qualquer início no log é assinante de antes
de 01/07/2024 e começa como ativo.

Regras checadas:
- `assinatura_iniciada` só sem assinatura ativa;
- `assinatura_expirada` só com assinatura ativa;
- os demais eventos (renovação, desligar/religar, carência, cancelamento, reembolso, chargeback) só com assinatura ativa.

In [19]:
ORDEM_NO_MESMO_INSTANTE = {"assinatura_iniciada": 0, "renovacao": 1, "renovacao_automatica_religada": 2,
                           "periodo_carencia": 3, "renovacao_automatica_desligada": 4, "cancelamento_web": 5,
                           "reembolso": 6, "chargeback": 7, "assinatura_expirada": 8}
sub = eventos[~eh_trial].copy()
sub["ordem"] = sub["evento"].map(ORDEM_NO_MESMO_INSTANTE)
sub = sub.sort_values(["user_id", "dh_utc", "ordem"], kind="stable")

delta = sub["evento"].map({"assinatura_iniciada": 1, "assinatura_expirada": -1}).fillna(0).astype(int)
primeiro = sub.groupby("user_id")["evento"].transform("first")
legado = (primeiro != "assinatura_iniciada").astype(int)  # já era assinante em 01/07/2024
sub["ativo_antes"] = legado + delta.groupby(sub["user_id"]).cumsum() - delta

regras = {
    "assinatura_iniciada com assinatura já ativa": (sub["evento"] == "assinatura_iniciada") & (sub["ativo_antes"] >= 1),
    "assinatura_expirada sem assinatura ativa": (sub["evento"] == "assinatura_expirada") & (sub["ativo_antes"] <= 0),
    "outro evento sem assinatura ativa": ~sub["evento"].isin(["assinatura_iniciada", "assinatura_expirada"]) & (sub["ativo_antes"] <= 0),
}
ordem = pd.Series({k: v.sum() for k, v in regras.items()}, name="linhas")
display(ordem.to_frame())
for k, v in list(regras.items())[:2]:
    achado("Consistência", k, f"{v.sum():,} linhas", v.sum(),
           "Nenhum se for zero; se não, olhar a sequência do usuário antes de reconstruir a situação")

,linhas
assinatura_iniciada com assinatura já ativa,0
assinatura_expirada sem assinatura ativa,0
outro evento sem assinatura ativa,398


Os eventos que aparecem sem assinatura ativa: o que vem antes deles e quanto tempo depois?

In [20]:
sub["anterior"] = sub.groupby("user_id")["evento"].shift()
sub["horas_apos_anterior"] = sub.groupby("user_id")["dh_utc"].diff().dt.total_seconds() / 3600
orfaos = sub[regras["outro evento sem assinatura ativa"]]
print("eventos:", orfaos["evento"].value_counts().to_dict())
print("evento anterior:", orfaos["anterior"].value_counts().to_dict())
print("horas depois do anterior:", orfaos["horas_apos_anterior"].describe().round(1).to_dict())
apos_expirar = int((orfaos["anterior"].isin(["assinatura_expirada", "cancelamento_web", "renovacao_automatica_desligada"])
                    & (orfaos["horas_apos_anterior"] <= 24)).sum())
achado("Consistência", "evento de assinatura registrado depois da expiração",
       f"{len(orfaos):,} linhas ({orfaos['evento'].value_counts().to_dict()}); {apos_expirar:,} chegam até 24h depois "
       "da expiração: são avisos atrasados da assinatura que acabou de terminar",
       len(orfaos),
       "Ao reconstruir, atribuir eventos de até 24h após a expiração à assinatura que terminou", status="esperado")

eventos: {'renovacao_automatica_desligada': 211, 'chargeback': 136, 'reembolso': 40, 'cancelamento_web': 11}
evento anterior: {'assinatura_expirada': 391, 'cancelamento_web': 4, 'renovacao_automatica_desligada': 3}
horas depois do anterior: {'count': 398.0, 'mean': 5.6, 'std': 4.2, 'min': 0.0, '25%': 2.0, '50%': 4.8, '75%': 8.5, 'max': 16.6}


In [21]:
# Os 'legados' (primeiro evento não é início) são mesmo assinantes de antes de 01/07/2024?
legados = sub.loc[legado.astype(bool), "user_id"].unique()
antigas = ass[(ass["inicio"] < INICIO_LOG) & (ass["data_fim"].isna() | (ass["data_fim"] >= INICIO_LOG))]
legado_sem_assinatura_antiga = (~pd.Series(legados).isin(antigas["user_id"])).sum()
print(f"usuários que começam o log já assinantes: {len(legados):,} | "
      f"sem assinatura anterior a 01/07/2024 em assinaturas.csv: {legado_sem_assinatura_antiga:,}")
achado("Consistência", "usuário que começa o log já assinante sem assinatura antiga em assinaturas.csv",
       f"{legado_sem_assinatura_antiga:,} de {len(legados):,} usuários", legado_sem_assinatura_antiga,
       "Se houver: o início dessas assinaturas se perdeu; usar inicio de assinaturas.csv")

usuários que começam o log já assinantes: 5,200 | sem assinatura anterior a 01/07/2024 em assinaturas.csv: 0


### 3.4 Situação final no log × `status_atual` em `assinaturas.csv`

In [22]:
final = (legado.groupby(sub["user_id"]).first() + delta.groupby(sub["user_id"]).sum()).rename("ativo_no_log")
ativos_retrato = set(ass.loc[ass["status_atual"] == "ativa", "user_id"])
comp = final.to_frame()
comp["ativa_no_retrato"] = comp.index.isin(ativos_retrato)
comp["ativo_no_log"] = comp["ativo_no_log"] >= 1
tab = pd.crosstab(comp["ativo_no_log"], comp["ativa_no_retrato"], margins=True)
display(tab)

ativos_fora_do_log = len(ativos_retrato - set(comp.index))
divergentes = int((comp["ativo_no_log"] != comp["ativa_no_retrato"]).sum())
print(f"assinantes ativos no retrato sem nenhum evento no log: {ativos_fora_do_log:,}")
print(f"ativos no log em 30/09: {comp['ativo_no_log'].sum():,} | ativos no retrato (sem teste): {len(ativos_retrato):,}")
achado("Consistência", "situação no fim do log diferente de status_atual",
       f"{divergentes:,} usuários divergem; {ativos_fora_do_log:,} ativos do retrato sem nenhum evento",
       divergentes + ativos_fora_do_log,
       "Se houver: decidir qual fonte vale para o corte e registrar em decisoes.md")

ativa_no_retrato,False,True,All
ativo_no_log,,,
False,17830,0,17830
True,0,14753,14753
All,17830,14753,32583


assinantes ativos no retrato sem nenhum evento no log: 0
ativos no log em 30/09: 14,753 | ativos no retrato (sem teste): 14,753


### 3.5 Plano e canal dentro da mesma assinatura

In [23]:
sub["seg"] = (sub["evento"] == "assinatura_iniciada").groupby(sub["user_id"]).cumsum()
por_seg = sub.groupby(["user_id", "seg"]).agg(planos=("plano", "nunique"), canais=("canal_cobranca", "nunique"))
troca_plano = int((por_seg["planos"] > 1).sum())
troca_canal = int((por_seg["canais"] > 1).sum())
print(f"assinaturas (no log) com mais de um plano: {troca_plano:,} | com mais de um canal: {troca_canal:,} | total: {len(por_seg):,}")
achado("Consistência", "assinatura que muda de plano no log", f"{troca_plano:,} de {len(por_seg):,} assinaturas", troca_plano,
       "Pode ser troca legítima (mensal para anual); usar o plano do evento, não o plano_atual do retrato, ao olhar o passado")
achado("Consistência", "assinatura que muda de canal no log", f"{troca_canal:,} de {len(por_seg):,} assinaturas", troca_canal,
       "Não é esperado sem nova assinatura; se houver, investigar")

assinaturas (no log) com mais de um plano: 0 | com mais de um canal: 0 | total: 32,583


### 3.6 Trial: todo teste grátis termina em expiração ou em assinatura?

In [24]:
tr = eventos[eventos["evento"].isin(["trial_iniciado", "trial_expirado", "assinatura_iniciada"])].copy()
tr["ordem"] = tr["evento"].map({"trial_iniciado": 0, "assinatura_iniciada": 1, "trial_expirado": 2})
tr = tr.sort_values(["user_id", "dh_utc", "ordem"], kind="stable")
tr["prox_evento"] = tr.groupby("user_id")["evento"].shift(-1)
tr["prox_dh"] = tr.groupby("user_id")["dh_utc"].shift(-1)
ini = tr[tr["evento"] == "trial_iniciado"].copy()
ini["dias_ate_prox"] = (ini["prox_dh"] - ini["dh_utc"]).dt.total_seconds() / 86400
ini["desfecho"] = ini["prox_evento"].fillna("(nenhum)")
display(ini["desfecho"].value_counts().to_frame("trials"))
display(ini.groupby("desfecho")["dias_ate_prox"].describe().round(2))

sem_desfecho = ini[ini["prox_evento"].isna()]
sem_desfecho_antigo = int((sem_desfecho["dh_br"] < FIM_CORTE - pd.Timedelta(days=8)).sum())
print(f"trials sem desfecho: {len(sem_desfecho):,}, dos quais {sem_desfecho_antigo:,} começaram mais de 8 dias antes do corte")
achado("Consistência", "trial sem desfecho (nem expiração nem assinatura) e iniciado há mais de 8 dias",
       f"{sem_desfecho_antigo:,} de {len(sem_desfecho):,} trials em aberto", sem_desfecho_antigo,
       "Trials dos últimos 7 dias estão em andamento; os mais antigos indicam evento faltando")

,trials
desfecho,
assinatura_iniciada,27383
trial_expirado,19143
(nenhum),470


,count,mean,std,min,25%,50%,75%,max
desfecho,,,,,,,,
(nenhum),0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
assinatura_iniciada,27383.0,7.0,0.29,6.3,6.80,7.0,7.21,7.70
trial_expirado,19143.0,7.0,0.29,6.3,6.79,7.0,7.21,7.71


trials sem desfecho: 470, dos quais 0 começaram mais de 8 dias antes do corte


## 4. Unicidade

O arquivo não tem identificador de evento. Um acontecimento repetido aparece como duas linhas iguais,
ou como a mesma coisa gravada duas vezes em horários quase iguais (por exemplo, uma em UTC e outra em horário de Brasília).

In [25]:
dup_exata = int(eventos.duplicated(subset=colunas).sum())
dup_chave = int(eventos.duplicated(subset=["user_id", "data_hora", "evento"]).sum())
dup_instante = int(eventos.duplicated(subset=["user_id", "dh_utc", "evento"]).sum())
print(f"linhas inteiras repetidas: {dup_exata:,}")
print(f"mesmo user_id + data_hora (texto) + evento: {dup_chave:,}")
print(f"mesmo user_id + instante (depois de acertar o fuso) + evento: {dup_instante:,}")
achado("Unicidade", "linhas inteiras repetidas", f"{dup_exata:,}", dup_exata, "Se houver: manter a primeira")
achado("Unicidade", "mesmo usuário, evento e instante (fuso acertado)", f"{dup_instante:,}", dup_instante,
       "Se houver: manter a primeira")

linhas inteiras repetidas: 0
mesmo user_id + data_hora (texto) + evento: 0
mesmo user_id + instante (depois de acertar o fuso) + evento: 0


In [26]:
# Quase-duplicatas: o mesmo evento do mesmo usuário em menos de 1 dia
UNICOS_POR_CICLO = ["assinatura_iniciada", "assinatura_expirada", "renovacao", "trial_iniciado", "trial_expirado"]
q = eventos[eventos["evento"].isin(UNICOS_POR_CICLO)].sort_values(["user_id", "evento", "dh_utc"])
q["gap_h"] = q.groupby(["user_id", "evento"])["dh_utc"].diff().dt.total_seconds() / 3600
perto = q[q["gap_h"] < 24]
display(perto.groupby("evento")["gap_h"].describe().round(2) if len(perto) else "nenhuma")
exatas_3h = int((perto["gap_h"] == 3).sum())
print(f"pares a menos de 24h: {len(perto):,} | exatamente 3h (sinal de fuso duplicado): {exatas_3h:,}")
achado("Unicidade", "mesmo evento do mesmo usuário a menos de 24h", f"{len(perto):,} linhas; {exatas_3h:,} a exatamente 3h",
       len(perto), "Se houver: tratar como duplicata e manter a primeira")

'nenhuma'

pares a menos de 24h: 0 | exatamente 3h (sinal de fuso duplicado): 0


## 5. Temporalidade

### 5.1 Período coberto e eventos depois do corte

O log deve ir de 01/07/2024 a 30/09/2026 (fim do dia, horário de Brasília).
A comparação abaixo mostra o efeito do fuso: lendo só o texto da data, um evento das 22h de 30/09 em Brasília,
gravado em UTC, parece ser de 01/10.

In [27]:
print(f"primeiro evento: {eventos['dh_br'].min()} | último: {eventos['dh_br'].max()} (horário de Brasília)")
antes = int((eventos["dh_br"] < INICIO_LOG).sum())
depois = int((eventos["dh_br"] >= FIM_CORTE).sum())
data_texto = pd.to_datetime(eventos["data_hora"].str[:10], format="%Y-%m-%d")
depois_texto = int((data_texto > CORTE).sum())
vira_dia = int((eventos["data_br"] != data_texto).sum())
print(f"antes de 01/07/2024: {antes:,} | depois do corte (Brasília): {depois:,} | 'depois do corte' lendo só o texto: {depois_texto:,}")
print(f"eventos cuja data muda ao converter para Brasília: {vira_dia:,}")
achado("Temporalidade", "eventos fora do período 01/07/2024 a 30/09/2026", f"{antes:,} antes e {depois:,} depois",
       antes + depois, "Se houver: excluir os posteriores ao corte")
achado("Temporalidade", "eventos que mudam de dia ao acertar o fuso",
       f"{vira_dia:,} linhas; lendo só o texto, {depois_texto:,} pareceriam posteriores ao corte",
       vira_dia, "Sempre usar a data convertida para Brasília para contar por dia ou mês")

primeiro evento: 2024-07-01 06:11:50 | último: 2026-09-30 22:59:33 (horário de Brasília)
antes de 01/07/2024: 0 | depois do corte (Brasília): 0 | 'depois do corte' lendo só o texto: 44
eventos cuja data muda ao converter para Brasília: 21,951


### 5.2 Volume por mês

Buracos (mês sem evento) ou saltos indicariam falha de envio.

In [28]:
mensal = pd.crosstab(eventos["mes_br"], eventos["evento"])
mensal["total"] = mensal.sum(axis=1)
meses_esperados = pd.period_range("2024-07", "2026-09", freq="M")
meses_faltando = meses_esperados.difference(mensal.index)
print(f"meses sem nenhum evento: {list(meses_faltando.astype(str))}")
display(mensal[["trial_iniciado", "assinatura_iniciada", "renovacao", "renovacao_automatica_desligada",
                "assinatura_expirada", "total"]])
variacao = mensal["total"].pct_change()
saltos = variacao[variacao.abs() > 0.5]
print("meses com variação acima de 50% sobre o anterior:", {str(k): f"{v:+.0%}" for k, v in saltos.items()})
achado("Temporalidade", "meses sem evento ou com salto acima de 50%",
       f"{len(meses_faltando)} meses vazios; {len(saltos)} saltos", len(meses_faltando) + len(saltos),
       "Se houver: conferir se é falha de envio antes de calcular churn do mês")

meses sem nenhum evento: []


evento,trial_iniciado,assinatura_iniciada,renovacao,renovacao_automatica_desligada,assinatura_expirada,total
mes_br,,,,,,
2024-07,1419,644,2670,496,106,5798
2024-08,1560,918,2917,632,277,6979
2024-09,1605,927,3114,668,329,7367
2024-10,1576,973,3467,711,353,7765
2024-11,1531,920,3660,683,348,7829
2024-12,1255,756,4030,724,385,7768
2025-01,2621,1423,4016,1090,545,10680
2025-02,1740,1134,3979,875,538,9143
2025-03,1841,1122,4681,909,559,9978


meses com variação acima de 50% sobre o anterior: {}


Referência do enunciado: em abril de 2026 a empresa contou 1.154 assinaturas perdidas. A série da empresa conta só quem
estava ativo no início do mês, então o número de expirações no log não precisa bater exatamente, mas deve estar perto.

In [29]:
print(f"assinatura_expirada em abril de 2026 (Brasília): {mensal.loc[pd.Period('2026-04', 'M'), 'assinatura_expirada']:,}")

assinatura_expirada em abril de 2026 (Brasília): 1,158


### 5.3 Ritmo das cobranças

Entre duas cobranças da mesma assinatura (início ou renovação), o mensal deve levar cerca de 1 mês e o anual cerca de 1 ano.

In [30]:
cob = sub[sub["evento"].isin(["assinatura_iniciada", "renovacao"])].copy()
cob["dias"] = cob.groupby(["user_id", "seg"])["dh_utc"].diff().dt.total_seconds() / 86400
cob = cob.dropna(subset=["dias"])
display(cob.groupby("plano")["dias"].describe(percentiles=[.01, .05, .5, .95, .99]).round(1))
FAIXA = {"mensal": (27, 33), "anual": (360, 370)}
fora_ritmo = cob.apply(lambda r: not (FAIXA[r["plano"]][0] <= r["dias"] <= FAIXA[r["plano"]][1])
                       if r["plano"] in FAIXA else True, axis=1)
print(f"renovações fora do ritmo esperado: {fora_ritmo.sum():,} de {len(cob):,}")
display(cob.loc[fora_ritmo].groupby("plano")["dias"].describe().round(1))

,count,mean,std,min,1%,5%,50%,95%,99%,max
plano,,,,,,,,,,
anual,4535.0,365.0,5.3,13.2,364.4,364.5,365.0,365.6,370.2,380.4
mensal,123718.0,30.0,1.1,8.8,28.1,29.5,30.0,30.5,32.7,45.6


renovações fora do ritmo esperado: 2,225 de 128,253


,count,mean,std,min,25%,50%,75%,max
plano,,,,,,,,
anual,58.0,364.6,47.2,13.2,370.3,371.0,372.1,380.4
mensal,2167.0,30.4,7.7,8.8,24.4,33.3,35.9,45.6


Hipótese: uma renovação atrasa alguns dias e a seguinte volta ao dia de sempre, deixando um ciclo longo e um curto em seguida.
A célula abaixo confere se houve `periodo_carencia` entre as duas cobranças, se ciclo longo e ciclo curto aparecem em pares
e em que canal isso acontece.

In [31]:
sub["carencias"] = (sub["evento"] == "periodo_carencia").groupby([sub["user_id"], sub["seg"]]).cumsum()
cob["carencia_entre"] = sub.loc[cob.index, "carencias"].groupby([cob["user_id"], cob["seg"]]).diff().gt(0)
cob["fora"] = fora_ritmo
cob["proximo_dias"] = cob.groupby(["user_id", "seg"])["dias"].shift(-1)
longos = cob[cob["fora"] & (cob["plano"] == "mensal") & (cob["dias"] > 33)]
compensa = (longos["dias"] + longos["proximo_dias"]).between(57, 63)
print(f"com período de carência entre as cobranças: {cob.loc[fora_ritmo, 'carencia_entre'].sum():,}")
print(f"ciclos mensais longos (>33 dias): {len(longos):,}, dos quais {compensa.sum():,} são compensados pelo ciclo seguinte "
      "(os dois somam ~60 dias)")
display(pd.crosstab(cob["canal_cobranca"], cob["fora"], normalize="index").round(3).rename(columns={True: "% fora do ritmo"})
        [["% fora do ritmo"]])
achado("Temporalidade", "renovação fora do intervalo esperado (mensal 27-33 dias, anual 360-370)",
       f"{fora_ritmo.sum():,} de {len(cob):,} cobranças; sem carência entre elas; ciclo longo seguido de ciclo curto "
       "que volta ao dia original, mais frequente no cartão",
       fora_ritmo.sum(),
       "Provável cobrança recusada e aprovada na nova tentativa; confirmar com pagamentos.csv (coluna tentativa)")

com período de carência entre as cobranças: 0
ciclos mensais longos (>33 dias): 1,156, dos quais 988 são compensados pelo ciclo seguinte (os dois somam ~60 dias)


fora,% fora do ritmo
canal_cobranca,
app_store,0.010
google_play,0.014
web_cartao,0.039
web_pix,0.014


## 6. Resumo

In [32]:
resumo = pd.DataFrame(achados)
resumo["% do arquivo"] = (100 * resumo["linhas_afetadas"] / N).round(2)
resumo.to_csv(SAIDAS / "qualidade_eventos_assinatura.csv", index=False)
resumo

,criterio,checagem,status,resultado,linhas_afetadas,tratamento_sugerido,% do arquivo
0,Completude,plano vazio em evento que não é trial_*,esperado,"6,734 linhas, todas renovacao_automatica_desli...",6734,Não é erro. Tratar como evento do trial (cance...,2.44
1,Completude,detalhe (motivo) ausente em assinatura_expirada,atenção,"11,759 de 17,830 expirações sem motivo (66%); ...",11759,Não usar detalhe como fonte de motivo; usar mo...,4.27
2,Completude,assinaturas sem evento de início (iniciadas a ...,ok,"0 de 27,383",0,Se houver: reconstruir o início pela data de i...,0.00
3,Completude,assinaturas encerradas sem evento de expiração,ok,"0 de 17,830",0,Se houver: usar data_fim de assinaturas.csv co...,0.00
4,Completude,evento de início sem assinatura em assinaturas...,ok,0 eventos,0,Se houver: investigar antes de usar; podem ser...,0.00
5,Completude,eventos de contas de teste,ok,0 eventos,0,"Se houver: excluir, como em assinaturas.csv",0.00
6,Validade,evento fora da lista do dicionário,ok,0 linhas,0,"Nenhum se for zero; se não, mapear para o valo...",0.00
7,Validade,plano fora da lista do dicionário,ok,0 linhas,0,"Nenhum se for zero; se não, mapear para o valo...",0.00
8,Validade,canal_cobranca fora da lista do dicionário,ok,0 linhas,0,"Nenhum se for zero; se não, mapear para o valo...",0.00
9,Validade,detalhe fora dos motivos de assinaturas.csv,ok,0 linhas,0,O dicionário não lista os valores de detalhe; ...,0.00


In [33]:
hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/brutos/ inalterado: True
